## Convert Mouse to Human Orthologs - COPY WITH SHORTER BATCHES higher delay

In [1]:
#!/usr/bin/env python3
"""
Convert mouse genes to human orthologs using biomaRt via mygene.info API

This script:
1. Loads mouse gene IDs from expression data
2. Queries mygene.info for human orthologs
3. Tracks conversion success/failure
4. Generates detailed conversion report
5. Creates converted expression files

No installation needed - uses public APIs!
"""

import os
import sys
import json
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from collections import defaultdict
import time
import datetime
import mygene


In [2]:
def query_mygene_orthologs_symbols(mouse_genes, batch_size=50, delay=0.3):
    """Query mygene.info for human orthologs using gene symbols"""
    import mygene
    
    orthologs = {}
    failed_genes = []
    
    overall_start = time.time()
    genes_list = list(mouse_genes)
    total_genes = len(genes_list)
    processed = 0
    batch_num = 0
    
    print(f"\nQuerying mygene.info for {total_genes} mouse genes (by symbol)...")
    print(f"(This may take a few minutes - API is rate-limited)")
    print(f"Using batch size: {batch_size} genes per request")
    
    mg = mygene.MyGeneInfo()
    
    i = 0
    while i < len(genes_list):
        batch_num += 1
        batch_start = time.time()
        batch = genes_list[i:i+batch_size]
        
        try:
            # Query batch of genes by SYMBOL
            gene_infos = mg.querymany(batch, scopes='symbol', species='mouse', 
                                      fields='homologene,pantherdb,symbol,entrezgene', 
                                      returnall=True)
            
            # Results are in 'out'
            if 'out' in gene_infos:
                for hit in gene_infos['out']:
                    # Skip if gene not found
                    if hit.get('notfound'):
                        failed_genes.append(hit.get('query', ''))
                        continue
                    
                    mouse_symbol = hit.get('symbol', hit.get('query', '')).upper()
                    human_orthologs = []
                    
                    # Method 1: Use homologene - extract human taxid (9606)
                    if 'homologene' in hit and 'genes' in hit['homologene']:
                        for gene_pair in hit['homologene']['genes']:
                            if len(gene_pair) >= 2:
                                taxid, entrez_id = gene_pair[0], gene_pair[1]
                                if taxid == 9606:  # Human
                                    # Query for human gene symbol
                                    try:
                                        human_gene_info = mg.query(f"entrezgene:{entrez_id}", 
                                                                   species='human', 
                                                                   fields='symbol')
                                        if human_gene_info and 'hits' in human_gene_info and len(human_gene_info['hits']) > 0:
                                            symbol = human_gene_info['hits'][0].get('symbol')
                                            if symbol:
                                                human_orthologs.append({
                                                    'source': 'HomoloGene',
                                                    'symbol': symbol,
                                                    'gene_id': entrez_id,
                                                    'taxid': 9606
                                                })
                                    except:
                                        pass
                    
                    # Method 2: Use pantherdb - extract HGNC info
                    if 'pantherdb' in hit and 'ortholog' in hit['pantherdb']:
                        for ortho in hit['pantherdb']['ortholog']:
                            if ortho.get('taxid') == 9606 and 'HGNC' in ortho:
                                human_orthologs.append({
                                    'source': 'PantherDB',
                                    'symbol': ortho.get('HGNC'),
                                    'taxid': 9606
                                })
                    
                    if human_orthologs and mouse_symbol:
                        orthologs[mouse_symbol] = human_orthologs
                    elif not human_orthologs and mouse_symbol:
                        failed_genes.append(mouse_symbol)
            
            processed += len(batch)
            batch_time = time.time() - batch_start
            elapsed = time.time() - overall_start
            print(f"  Batch {batch_num}: {processed}/{total_genes} genes | Batch time: {batch_time:.2f}s | Total time: {elapsed:.1f}s")
            
            time.sleep(delay)
            
        except Exception as e:
            print(f"  ⚠️  Error on batch {batch_num}: {str(e)}")
            failed_genes.extend(batch)
            processed += len(batch)
        
        i += batch_size
    
    return orthologs, failed_genes

def load_expression_data(expr_file):
    """Load expression CSV file"""
    print(f"Loading {expr_file}...")
    df = pd.read_csv(expr_file, index_col=0)
    print(f"  Loaded {df.shape[0]} genes × {df.shape[1]} cell types")
    return df

def convert_expression_data(expr_df, orthologs, tissue_name):
    """Convert mouse genes to human orthologs in expression data"""
    print(f"\nConverting {tissue_name} expression data...")
    
    mouse_genes = set(expr_df.index.astype(str))
    gene_mapping = {}
    converted_genes = set()
    unconverted_genes = set()
    
    for mouse_gene in mouse_genes:
        mouse_gene_upper = str(mouse_gene).upper()
        
        # Try exact match first
        if mouse_gene_upper in orthologs:
            human_orthologs = orthologs[mouse_gene_upper]
            if human_orthologs:
                # Prioritize PantherDB over HomoloGene
                human_gene = None
                for ortho in human_orthologs:
                    if ortho.get('source') == 'PantherDB':
                        human_gene = ortho.get('symbol')
                        break
                if not human_gene:
                    human_gene = human_orthologs[0].get('symbol')
                
                if human_gene:
                    gene_mapping[mouse_gene] = str(human_gene)
                    converted_genes.add(mouse_gene)
                else:
                    unconverted_genes.add(mouse_gene)
            else:
                unconverted_genes.add(mouse_gene)
        else:
            unconverted_genes.add(mouse_gene)
    
    # Rename index
    expr_converted = expr_df.copy()
    expr_converted.index = expr_converted.index.astype(str).map(lambda x: gene_mapping.get(x, None))
    
    # Remove unconverted genes
    expr_converted = expr_converted[expr_converted.index.notna()]
    
    # Handle duplicates
    if not expr_converted.index.is_unique:
        n_before = len(expr_converted)
        expr_converted = expr_converted.groupby(level=0).mean()
        n_after = len(expr_converted)
        n_duplicates = n_before - n_after
        print(f"  Merged {n_duplicates} duplicate human genes")
    
    stats = {
        'tissue': tissue_name,
        'n_mouse_genes': len(mouse_genes),
        'n_converted': len(converted_genes),
        'n_unconverted': len(unconverted_genes),
        'n_human_genes': len(expr_converted),
        'conversion_rate': len(converted_genes) / len(mouse_genes) * 100 if len(mouse_genes) > 0 else 0,
        'retention_rate': len(expr_converted) / len(mouse_genes) * 100 if len(mouse_genes) > 0 else 0,
    }
    
    print(f"  Mouse genes: {stats['n_mouse_genes']}")
    print(f"  Converted: {stats['n_converted']} ({stats['conversion_rate']:.1f}%)")
    print(f"  Unconverted: {stats['n_unconverted']} ({100-stats['conversion_rate']:.1f}%)")
    print(f"  Final human genes: {stats['n_human_genes']} ({stats['retention_rate']:.1f}% of original)")
    
    return expr_converted, stats, gene_mapping, unconverted_genes

def generate_conversion_report(all_stats, all_unconverted, all_mappings, output_dir):
    """Generate detailed conversion report"""
    os.makedirs(output_dir, exist_ok=True)
    
    print("\n" + "="*80)
    print("CONVERSION REPORT")
    print("="*80)
    
    # Summary statistics
    stats_df = pd.DataFrame(all_stats)
    print("\nPer-Tissue Summary:")
    print(stats_df.to_string(index=False))
    
    # Overall statistics
    total_mouse = stats_df['n_mouse_genes'].sum()
    total_converted = stats_df['n_converted'].sum()
    total_human = stats_df['n_human_genes'].sum()
    
    print(f"\nOverall Statistics:")
    print(f"  Total mouse genes (across all tissues): {total_mouse}")
    print(f"  Total converted: {total_converted} ({total_converted/total_mouse*100:.1f}%)")
    print(f"  Total human genes (after deduplication): {total_human}")
    print(f"  Average retention: {stats_df['retention_rate'].mean():.1f}%")
    
    # Save detailed report
    report_file = os.path.join(output_dir, 'conversion_report.txt')
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MOUSE-TO-HUMAN ORTHOLOG CONVERSION REPORT\n")
        f.write("="*80 + "\n\n")
        
        f.write("SUMMARY STATISTICS\n")
        f.write("-"*80 + "\n")
        f.write(stats_df.to_string(index=False))
        f.write(f"\n\nOverall Statistics:\n")
        f.write(f"  Total mouse genes: {total_mouse}\n")
        f.write(f"  Total converted: {total_converted} ({total_converted/total_mouse*100:.1f}%)\n")
        f.write(f"  Total human genes: {total_human}\n")
        f.write(f"  Average retention: {stats_df['retention_rate'].mean():.1f}%\n")
    
    print(f"\n✓ Report saved to {report_file}")
    
    # Save statistics CSV
    stats_csv = os.path.join(output_dir, 'conversion_statistics.csv')
    stats_df.to_csv(stats_csv, index=False)
    print(f"✓ Statistics saved to {stats_csv}")
    
    return stats_df

In [3]:
# Main execution

expr_dir = '../R/processed'
output_dir = './converted_expression'
report_dir = './conversion_report'
tissues = ['drgn', 'drgnn', 'tgn', 'tgnn']

os.makedirs(output_dir, exist_ok=True)
os.makedirs(report_dir, exist_ok=True)

print("="*80)
print("MOUSE-TO-HUMAN ORTHOLOG CONVERSION (ENTREZ ID VERSION)")
print("="*80)

# Step 1: Collect all mouse genes
print("\nStep 1: Collecting all mouse genes from expression data...")
all_mouse_genes = set()
tissue_genes = {}

for tissue in tissues:
    expr_file = os.path.join(expr_dir, f'{tissue}_celltype_avg_expression.csv')
    if not os.path.exists(expr_file):
        print(f"  ⚠️  File not found: {expr_file}")
        continue

    expr_df = load_expression_data(expr_file)
    # Convert to string and uppercase
    tissue_genes[tissue] = set(expr_df.index.astype(str).str.upper())
    all_mouse_genes.update(tissue_genes[tissue])

print(f"  Total unique mouse genes: {len(all_mouse_genes)}")

MOUSE-TO-HUMAN ORTHOLOG CONVERSION (ENTREZ ID VERSION)

Step 1: Collecting all mouse genes from expression data...
Loading ../R/processed/drgn_celltype_avg_expression.csv...
  Loaded 112469 genes × 18 cell types
Loading ../R/processed/drgnn_celltype_avg_expression.csv...
  Loaded 112469 genes × 7 cell types
Loading ../R/processed/tgn_celltype_avg_expression.csv...
  Loaded 67033 genes × 15 cell types
Loading ../R/processed/tgnn_celltype_avg_expression.csv...
  Loaded 67033 genes × 11 cell types
  Total unique mouse genes: 121900


In [4]:
# Filter out junk genes that won't convert
def is_valid_gene_symbol(gene):
    """Keep only standard gene symbols, filter out junk"""
    gene = str(gene).strip().upper()
    
    skip_patterns = [
        'ENSMFAG',       # Manatee Ensembl IDs
        'ENSDARG',       # Zebrafish
        'ENSRNOG',       # Rat Ensembl IDs
        'MIR',           # microRNAs
        'LINC',          # Long non-coding
        '-PS',           # Pseudogenes
        'AC0', 'AL0', 'AP0',  # Predicted/contig sequences
        'RP23', 'RP24',  # BAC clones
        'U6.',           # Small RNAs
    ]
    
    for pattern in skip_patterns:
        if pattern in gene:
            return False
    
    return len(gene) > 0 and gene[0].isalpha()

filtered_genes = {g for g in all_mouse_genes if is_valid_gene_symbol(g)}

print(f"Original genes: {len(all_mouse_genes)}")
print(f"Filtered genes: {len(filtered_genes)}")
print(f"Removed: {len(all_mouse_genes) - len(filtered_genes)} junk genes")

Original genes: 121900
Filtered genes: 89306
Removed: 32594 junk genes


In [5]:
# Step 2: Query orthologs ONCE for all unique genes
print("\nStep 2: Querying mygene.info for human orthologs...")
now = datetime.datetime.now()
print("Started at: " + now.strftime("%Y-%m-%d %H:%M:%S"))

orthologs, failed_genes = query_mygene_orthologs_symbols(list(filtered_genes))

now = datetime.datetime.now()
print("Finished at: " + now.strftime("%Y-%m-%d %H:%M:%S"))

print(f"\n✓ Found orthologs for {len(orthologs)} genes")
print(f"✗ Failed to find orthologs for {len(failed_genes)} genes")

orthologs_file = os.path.join(output_dir, 'mouse_to_human_orthologs.json')
with open(orthologs_file, 'w') as f:
    json.dump(orthologs, f)
print(f"\n✓ Saved {len(orthologs)} orthologs to {orthologs_file}")

# Step 3: Convert expression data
print("\nStep 3: Converting expression data...")
all_stats = []
all_unconverted = {}
all_mappings = {}

for tissue in tissues:
    expr_file = os.path.join(expr_dir, f'{tissue}_celltype_avg_expression.csv')
    if not os.path.exists(expr_file):
        continue

    expr_df = load_expression_data(expr_file)
    expr_converted, stats, mapping, unconverted = convert_expression_data(
        expr_df, orthologs, tissue
    )
    all_stats.append(stats)
    all_unconverted[tissue] = unconverted
    all_mappings[tissue] = mapping

    # Save converted expression
    output_file = os.path.join(output_dir, f'{tissue}_celltype_avg_expression_human.csv')
    expr_converted.to_csv(output_file)
    print(f"  ✓ Saved to {output_file}")

    # Also convert detection rate if available
    detection_file = os.path.join(expr_dir, f'{tissue}_celltype_detection_rate.csv')
    if os.path.exists(detection_file):
        detection_df = load_expression_data(detection_file)
        detection_converted, _, _, _ = convert_expression_data(
            detection_df, orthologs, f"{tissue} (detection rate)"
        )
        detection_output = os.path.join(output_dir, f'{tissue}_celltype_detection_rate_human.csv')
        detection_converted.to_csv(detection_output)
        print(f"  ✓ Saved detection rate to {detection_output}")

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed



Step 2: Querying mygene.info for human orthologs...
Started at: 2025-12-28 13:46:01

Querying mygene.info for 89306 mouse genes (by symbol)...
(This may take a few minutes - API is rate-limited)
Using batch size: 50 genes per request


4 input query terms found dup hits:	[('GM42064', 2), ('GM6212', 2), ('GM13005', 2), ('GM9218', 2)]
28 input query terms found no hit:	['OLFR1458', 'AC113167.1', 'GM26530', 'AI747448', 'SNORD36B', 'NBPF12', 'KIAA1161', 'Z95115.2', 'AL1
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


  Batch 1: 50/89306 genes | Batch time: 3.58s | Total time: 3.6s


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
6 input query terms found dup hits:	[('GM14004', 2), ('GM28153', 2), ('GM19898', 2), ('GM5831', 2), ('GM34838', 2), ('GM5855', 2)]
21 input query terms found no hit:	['GM26804', 'AL450992.1', 'METAZOA-SRP.1929', 'LOC114677663', 'METAZOA-SRP.1956', 'RMDN2-AS1', 'LOC1
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string form

  Batch 2: 100/89306 genes | Batch time: 2.96s | Total time: 6.8s


... (2,118 outputs left out)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
2 input query terms found dup hits:	[('GM29530', 2), ('GM13547', 2)]
29 input query terms found no hit:	['LOC114678641', 'AL592293.2', 'LOC100423539', 'SNORD62A', 'AL358472.3', 'C8ORF34', 'X4933406J10RIK'
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already 

  Batch 1786: 89300/89306 genes | Batch time: 3.31s | Total time: 5944.8s


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
3 input query terms found no hit:	['RABL2A', 'RFWD2', 'LOC105376737']
Input sequence provided is already in string format. No operation performed


  Batch 1787: 89306/89306 genes | Batch time: 1.37s | Total time: 5946.5s
Finished at: 2025-12-28 15:25:07

✓ Found orthologs for 18875 genes
✗ Failed to find orthologs for 77107 genes

✓ Saved 18875 orthologs to ./converted_expression/mouse_to_human_orthologs.json

Step 3: Converting expression data...
Loading ../R/processed/drgn_celltype_avg_expression.csv...
  Loaded 112469 genes × 18 cell types

Converting drgn expression data...
  Merged 1840 duplicate human genes
  Mouse genes: 112469
  Converted: 18859 (16.8%)
  Unconverted: 93610 (83.2%)
  Final human genes: 17019 (15.1% of original)
  ✓ Saved to ./converted_expression/drgn_celltype_avg_expression_human.csv
Loading ../R/processed/drgn_celltype_detection_rate.csv...
  Loaded 114660 genes × 18 cell types

Converting drgn (detection rate) expression data...
  Merged 3360 duplicate human genes
  Mouse genes: 112469
  Converted: 18859 (16.8%)
  Unconverted: 93610 (83.2%)
  Final human genes: 17019 (15.1% of original)
  ✓ Saved detec

In [ ]:
# Step 3.5: Deduplicate across tissues
print("\nStep 3.5: Deduplicating genes across tissues...")

all_expr_dfs = []
for tissue in tissues:
    output_file = os.path.join(output_dir, f'{tissue}_celltype_avg_expression_human.csv')
    if os.path.exists(output_file):
        expr_df = pd.read_csv(output_file, index_col=0)
        all_expr_dfs.append(expr_df)

if all_expr_dfs:
    # Concatenate all tissues
    expr_combined = pd.concat(all_expr_dfs, axis=1)
    
    # Deduplicate by taking mean across duplicate human genes
    n_before = len(expr_combined)
    expr_combined = expr_combined.groupby(level=0).mean()
    n_after = len(expr_combined)
    n_duplicates = n_before - n_after
    
    print(f"  Before deduplication: {n_before} genes")
    print(f"  After deduplication: {n_after} genes")
    print(f"  Merged: {n_duplicates} duplicate human genes")
    
    # Save deduplicated combined file
    combined_output = os.path.join(output_dir, 'all_tissues_combined_human.csv')
    expr_combined.to_csv(combined_output)
    print(f"  ✓ Saved combined deduplicated file to {combined_output}")



Step 3.5: Deduplicating genes across tissues...
  Before deduplication: 17030 genes
  After deduplication: 17030 genes
  Merged: 0 duplicate human genes
  ✓ Saved combined deduplicated file to ./converted_expression/all_tissues_combined_human.csv


In [ ]:
# Step 4: Generate report
print("\nStep 4: Generating conversion report...")
stats_df = generate_conversion_report(all_stats, all_unconverted, all_mappings, report_dir)

print("\n" + "="*80)
print("CONVERSION COMPLETE!")
print("="*80)
print(f"\nNext steps:")
print(f"1. Review conversion report in: {report_dir}/")
print(f"2. Use converted files from: {output_dir}/")
print(f"3. Run: python build_multiplex_interactome.py --human-expr-file {output_dir}/all_tissues_combined_human.csv")
print("="*80)


Step 4: Generating conversion report...

CONVERSION REPORT

Per-Tissue Summary:
tissue  n_mouse_genes  n_converted  n_unconverted  n_human_genes  conversion_rate  retention_rate
  drgn         112469        18859          93610          17019        16.768176       15.132170
 drgnn         112469        18859          93610          17019        16.768176       15.132170
   tgn          67033        19135          47898          17001        28.545642       25.362135
  tgnn          67033        19135          47898          17001        28.545642       25.362135

Overall Statistics:
  Total mouse genes (across all tissues): 359004
  Total converted: 75988 (21.2%)
  Total human genes (after deduplication): 68040
  Average retention: 20.2%

✓ Report saved to ./conversion_report/conversion_report.txt
✓ Statistics saved to ./conversion_report/conversion_statistics.csv

CONVERSION COMPLETE!

Next steps:
1. Review conversion report in: ./conversion_report/
2. Use converted files from: ./co

# OLD THAT PRODUCED NUMERIC GENE IDS

In [2]:
def query_mygene_orthologs(mouse_genes, batch_size=50, delay=0.3):
    """Query mygene.info for human orthologs using the Python package."""
    import mygene
    
    orthologs = {}
    failed_genes = []
    
    overall_start = time.time()
    genes_list = list(mouse_genes)
    total_genes = len(genes_list)
    processed = 0
    batch_num = 0
    
    print(f"\nQuerying mygene.info for {total_genes} mouse genes...")
    print(f"(This may take a few minutes - API is rate-limited)")
    print(f"Using batch size: {batch_size} genes per request")
    
    mg = mygene.MyGeneInfo()
    
    # Build a cache of Entrez ID -> Human Symbol mappings
    entrez_to_symbol_cache = {}
    
    i = 0
    while i < len(genes_list):
        batch_num += 1
        batch_start = time.time()
        batch = genes_list[i:i+batch_size]
        
        try:
            # Query batch of genes by symbol
            gene_infos = mg.querymany(batch, scopes='symbol', species='mouse', 
                                      fields='homologene,pantherdb', returnall=True)
            
            # Results are in 'out', not 'hits'
            if 'out' in gene_infos:
                for hit in gene_infos['out']:
                    # Skip if gene not found
                    if hit.get('notfound'):
                        failed_genes.append(hit.get('query', ''))
                        continue
                    
                    mouse_symbol = hit.get('query', '').upper()
                    human_orthologs = []
                    human_entrez_ids = []
                    
                    # Method 1: Use homologene - extract human taxid (9606)
                    if 'homologene' in hit and 'genes' in hit['homologene']:
                        for gene_pair in hit['homologene']['genes']:
                            if len(gene_pair) >= 2:
                                taxid, entrez_id = gene_pair[0], gene_pair[1]
                                if taxid == 9606:  # Human
                                    human_entrez_ids.append(entrez_id)
                    
                    # Method 2: Use pantherdb - extract HGNC info
                    if 'pantherdb' in hit and 'ortholog' in hit['pantherdb']:
                        for ortho in hit['pantherdb']['ortholog']:
                            if ortho.get('taxid') == 9606 and 'HGNC' in ortho:
                                human_orthologs.append({
                                    'source': 'PantherDB',
                                    'symbol': ortho.get('HGNC'),
                                    'taxid': 9606
                                })
                    
                    # Convert Entrez IDs to symbols
                    for entrez_id in human_entrez_ids:
                        if entrez_id not in entrez_to_symbol_cache:
                            try:
                                # Query for human gene symbol
                                human_gene_info = mg.query(f"entrezgene:{entrez_id}", species='human', fields='symbol')
                                if human_gene_info and 'hits' in human_gene_info and len(human_gene_info['hits']) > 0:
                                    symbol = human_gene_info['hits'][0].get('symbol')
                                    entrez_to_symbol_cache[entrez_id] = symbol
                                else:
                                    entrez_to_symbol_cache[entrez_id] = None
                            except:
                                entrez_to_symbol_cache[entrez_id] = None
                        
                        symbol = entrez_to_symbol_cache[entrez_id]
                        if symbol:
                            human_orthologs.append({
                                'source': 'HomoloGene',
                                'symbol': symbol,
                                'gene_id': entrez_id,
                                'taxid': 9606
                            })
                    
                    if human_orthologs and mouse_symbol:
                        orthologs[mouse_symbol] = human_orthologs
                    elif not human_orthologs and mouse_symbol:
                        failed_genes.append(mouse_symbol)
            
            processed += len(batch)
            batch_time = time.time() - batch_start
            elapsed = time.time() - overall_start
            print(f"  Batch {batch_num}: {processed}/{total_genes} genes | Batch time: {batch_time:.2f}s | Total time: {elapsed:.1f}s")
            
            time.sleep(delay)
            
        except Exception as e:
            print(f"  ⚠️  Error on batch {batch_num}: {str(e)}")
            failed_genes.extend(batch)
            processed += len(batch)
        
        i += batch_size
    
    return orthologs, failed_genes

def load_expression_data(expr_file):
    """Load expression CSV file"""
    print(f"Loading {expr_file}...")
    df = pd.read_csv(expr_file, index_col=0)
    print(f"  Loaded {df.shape[0]} genes × {df.shape[1]} cell types")
    return df

def convert_expression_data(expr_df, orthologs, tissue_name):
    """Convert mouse genes to human orthologs in expression data"""
    print(f"\nConverting {tissue_name} expression data...")
    
    mouse_genes = set(expr_df.index)
    gene_mapping = {}
    converted_genes = set()
    unconverted_genes = set()
    
    for mouse_gene in mouse_genes:
        mouse_gene_upper = mouse_gene.upper()
        if mouse_gene_upper in orthologs:
            # Extract human gene symbol from the list of orthologs
            # Use the first one (prioritize PantherDB over HomoloGene if both exist)
            human_orthologs = orthologs[mouse_gene_upper]
            if human_orthologs:
                # Get symbol - should now be populated
                human_gene = human_orthologs[0].get('symbol')
                if human_gene:
                    gene_mapping[mouse_gene] = str(human_gene)
                    converted_genes.add(mouse_gene)
        else:
            unconverted_genes.add(mouse_gene)
    
    # Rename index
    expr_converted = expr_df.copy()
    expr_converted.index = expr_converted.index.map(lambda x: gene_mapping.get(x, None))
    
    # Remove unconverted genes
    expr_converted = expr_converted[expr_converted.index.notna()]
    
    # Handle duplicates
    if not expr_converted.index.is_unique:
        n_before = len(expr_converted)
        expr_converted = expr_converted.groupby(level=0).mean()
        n_after = len(expr_converted)
        n_duplicates = n_before - n_after
        print(f"  Merged {n_duplicates} duplicate human genes")
    
    stats = {
        'tissue': tissue_name,
        'n_mouse_genes': len(mouse_genes),
        'n_converted': len(converted_genes),
        'n_unconverted': len(unconverted_genes),
        'n_human_genes': len(expr_converted),
        'conversion_rate': len(converted_genes) / len(mouse_genes) * 100,
        'retention_rate': len(expr_converted) / len(mouse_genes) * 100,
    }
    
    print(f"  Mouse genes: {stats['n_mouse_genes']}")
    print(f"  Converted: {stats['n_converted']} ({stats['conversion_rate']:.1f}%)")
    print(f"  Unconverted: {stats['n_unconverted']} ({100-stats['conversion_rate']:.1f}%)")
    print(f"  Final human genes: {stats['n_human_genes']} ({stats['retention_rate']:.1f}% of original)")
    
    return expr_converted, stats, gene_mapping, unconverted_genes

def generate_conversion_report(all_stats, all_unconverted, all_mappings, output_dir):
    """
    Generate detailed conversion report
    
    Parameters:
    - all_stats: list of conversion statistics dicts
    - all_unconverted: dict mapping tissue -> set of unconverted genes
    - all_mappings: dict mapping tissue -> gene_mapping dict
    - output_dir: directory to save reports
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("\n" + "="*80)
    print("CONVERSION REPORT")
    print("="*80)
    
    # Summary statistics
    stats_df = pd.DataFrame(all_stats)
    print("\nPer-Tissue Summary:")
    print(stats_df.to_string(index=False))
    
    # Overall statistics
    total_mouse = stats_df['n_mouse_genes'].sum()
    total_converted = stats_df['n_converted'].sum()
    total_human = stats_df['n_human_genes'].sum()
    
    print(f"\nOverall Statistics:")
    print(f"  Total mouse genes (across all tissues): {total_mouse}")
    print(f"  Total converted: {total_converted} ({total_converted/total_mouse*100:.1f}%)")
    print(f"  Total human genes (after deduplication): {total_human}")
    print(f"  Average retention: {stats_df['retention_rate'].mean():.1f}%")
    
    # Identify genes lost per tissue
    print(f"\nGene Loss by Tissue:")
    for tissue, unconverted in all_unconverted.items():
        print(f"  {tissue}: {len(unconverted)} genes lost")
    
    # Save detailed report
    report_file = os.path.join(output_dir, 'conversion_report.txt')
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MOUSE-TO-HUMAN ORTHOLOG CONVERSION REPORT\n")
        f.write("="*80 + "\n\n")
        
        f.write("SUMMARY STATISTICS\n")
        f.write("-"*80 + "\n")
        f.write(stats_df.to_string(index=False))
        f.write(f"\n\nOverall Statistics:\n")
        f.write(f"  Total mouse genes: {total_mouse}\n")
        f.write(f"  Total converted: {total_converted} ({total_converted/total_mouse*100:.1f}%)\n")
        f.write(f"  Total human genes: {total_human}\n")
        f.write(f"  Average retention: {stats_df['retention_rate'].mean():.1f}%\n")
        
        f.write(f"\n\nGENE LOSS BY TISSUE\n")
        f.write("-"*80 + "\n")
        for tissue, unconverted in all_unconverted.items():
            f.write(f"\n{tissue}: {len(unconverted)} genes lost\n")
            if len(unconverted) <= 50:
                for gene in sorted(unconverted):
                    f.write(f"  - {gene}\n")
            else:
                for gene in sorted(list(unconverted)[:50]):
                    f.write(f"  - {gene}\n")
                f.write(f"  ... and {len(unconverted) - 50} more\n")
        
        f.write(f"\n\nCONVERSION MAPPINGS\n")
        f.write("-"*80 + "\n")
        for tissue, mapping in all_mappings.items():
            f.write(f"\n{tissue}: {len(mapping)} genes mapped\n")
            if len(mapping) <= 20:
                for mouse_gene, human_gene in sorted(mapping.items()):
                    f.write(f"  {mouse_gene} -> {human_gene}\n")
            else:
                for mouse_gene, human_gene in sorted(list(mapping.items())[:20]):
                    f.write(f"  {mouse_gene} -> {human_gene}\n")
                f.write(f"  ... and {len(mapping) - 20} more\n")
    
    print(f"\n✓ Report saved to {report_file}")
    
    # Save statistics CSV
    stats_csv = os.path.join(output_dir, 'conversion_statistics.csv')
    stats_df.to_csv(stats_csv, index=False)
    print(f"✓ Statistics saved to {stats_csv}")
    
    # Save unconverted genes per tissue
    for tissue, unconverted in all_unconverted.items():
        unconverted_file = os.path.join(output_dir, f'{tissue}_unconverted_genes.txt')
        with open(unconverted_file, 'w') as f:
            for gene in sorted(unconverted):
                f.write(f"{gene}\n")
        print(f"✓ Unconverted genes for {tissue} saved to {unconverted_file}")
    
    return stats_df

In [3]:
expr_dir = '../R/processed'
output_dir = 'converted_expression'
report_dir = 'conversion_report'
tissues = ['drgn', 'drgnn', 'tgn', 'tgnn']

os.makedirs(output_dir, exist_ok=True)
os.makedirs(report_dir, exist_ok=True)

print("="*80)
print("MOUSE-TO-HUMAN ORTHOLOG CONVERSION")
print("="*80)

# Step 1: Collect all mouse genes
print("\nStep 1: Collecting all mouse genes from expression data...")
all_mouse_genes = set()
tissue_genes = {}

for tissue in tissues:
    expr_file = os.path.join(expr_dir, f'{tissue}_celltype_avg_expression.csv')
    if not os.path.exists(expr_file):
        print(f"  ⚠️  File not found: {expr_file}")
        continue

    expr_df = load_expression_data(expr_file)
    # Normalize to uppercase immediately
    tissue_genes[tissue] = set(expr_df.index.str.upper())
    all_mouse_genes.update(tissue_genes[tissue])

print(f"  Total unique mouse genes: {len(all_mouse_genes)}")

MOUSE-TO-HUMAN ORTHOLOG CONVERSION

Step 1: Collecting all mouse genes from expression data...
Loading ../R/processed/drgn_celltype_avg_expression.csv...
  Loaded 112469 genes × 18 cell types
Loading ../R/processed/drgnn_celltype_avg_expression.csv...
  Loaded 112469 genes × 7 cell types
Loading ../R/processed/tgn_celltype_avg_expression.csv...
  Loaded 67033 genes × 15 cell types
Loading ../R/processed/tgnn_celltype_avg_expression.csv...
  Loaded 67033 genes × 11 cell types
  Total unique mouse genes: 121900


In [4]:
# Filter to only standard gene symbols (what mygene.info actually works with)
def is_valid_gene_symbol(gene):
    """Keep only standard gene symbols, filter out junk"""
    gene = str(gene).strip()
    
    # Skip these patterns - they won't convert well
    skip_patterns = [
        'LOC',           # Unannotated loci
        'ENSRNOG',       # Rat Ensembl IDs
        'ENSMFAG',       # Manatee Ensembl IDs
        'ENSDARG',       # Zebrafish
        'MIR',           # microRNAs
        'LINC',          # Long non-coding
        '-PS',           # Pseudogenes
        'AC0', 'AL0', 'AP0',  # Predicted/contig sequences
        'RP23', 'RP24',  # BAC clones
        'GM',            # Predicted genes (low confidence)
        'RIK',           # Predicted genes
        'U6.',           # Small RNAs
    ]
    
    for pattern in skip_patterns:
        if pattern in gene.upper():
            return False
    
    return len(gene) > 0 and gene[0].isalpha()

# Filter your genes
filtered_genes = {g for g in all_mouse_genes if is_valid_gene_symbol(g)}

print(f"Original genes: {len(all_mouse_genes)}")
print(f"Filtered genes: {len(filtered_genes)}")
print(f"Removed: {len(all_mouse_genes) - len(filtered_genes)} junk genes")

Original genes: 121900
Filtered genes: 40681
Removed: 81219 junk genes


In [5]:
# Get the current date and time
now = datetime.datetime.now()
# Display a message indicating what is being printed
print("Started at: ")
# Print the current date and time in a specific format
print(now.strftime("%Y-%m-%d %H:%M:%S"))

# Step 2: Query orthologs ONCE for all unique genes
print("\nStep 2: Querying mygene.info for human orthologs...")
orthologs, failed_genes = query_mygene_orthologs(list(filtered_genes))

now = datetime.datetime.now()
# Display a message indicating what is being printed
print("Finished at: ")
# Print the current date and time in a specific format
print(now.strftime("%Y-%m-%d %H:%M:%S"))

print(f"\n✓ Found orthologs for {len(orthologs)} genes")
print(f"✗ Failed to find orthologs for {len(failed_genes)} genes")

orthologs_file = os.path.join(output_dir, 'mouse_to_human_orthologs.json')
with open(orthologs_file, 'w') as f:
    json.dump(orthologs, f)
print(f"\n✓ Saved {len(orthologs)} orthologs to {orthologs_file}")

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Started at: 
2025-12-27 15:05:51

Step 2: Querying mygene.info for human orthologs...

Querying mygene.info for 40681 mouse genes...
(This may take a few minutes - API is rate-limited)
Using batch size: 50 genes per request


1 input query terms found dup hits:	[('SNORD99', 2)]
17 input query terms found no hit:	['CHD3.1', 'TSPY.PS', 'VPREB2', 'AC100826.1', 'Z82206.1', 'METAZOA-SRP.444', 'AL133215.2', 'IGLVIVOR
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format

  Batch 1: 50/40681 genes | Batch time: 6.74s | Total time: 6.7s


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('IGHV5-11', 2)]
27 input query terms found no hit:	['METAZOA-SRP.174', 'AC108457.1', 'AABR07062570.1', 'AL837512.1', 'CCDC109B', 'AC108136.1', 'SNORA10
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string forma

  Batch 2: 100/40681 genes | Batch time: 4.55s | Total time: 11.6s


... (1,620 outputs left out)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
24 input query terms found no hit:	['METAZOA-SRP.395', 'THEGL', 'CRE', 'AC144833.1', 'AL590997.2', 'C12ORF74', 'MATR3-1', 'ZNF702P', 'A
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is a

  Batch 813: 40650/40681 genes | Batch time: 5.36s | Total time: 4331.1s


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
9 input query terms found no hit:	['CT030722.1', 'AC158355.1', 'AL589669.1', 'AC138646.1', 'AL121782.1', 'STK17A', 'OLFR642', 'AC16361
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is al

  Batch 814: 40681/40681 genes | Batch time: 3.99s | Total time: 4335.4s
Finished at: 
2025-12-27 16:18:07

✓ Found orthologs for 18651 genes
✗ Failed to find orthologs for 22566 genes

✓ Saved 18651 orthologs to converted_expression/mouse_to_human_orthologs.json


In [6]:
# Display a message indicating what is being printed
print("Current date and time : ")

# Print the current date and time in a specific format
print(now.strftime("%Y-%m-%d %H:%M:%S"))
# Step 3: Convert expression data (reuse the same orthologs dict)
print("\nStep 3: Converting expression data...")
all_stats = []
all_unconverted = {}
all_mappings = {}

for tissue in tissues:
    expr_file = os.path.join(expr_dir, f'{tissue}_celltype_avg_expression.csv')
    if not os.path.exists(expr_file):
        continue

    expr_df = load_expression_data(expr_file)
    # Pass the pre-computed orthologs dict - no re-querying!
    expr_converted, stats, mapping, unconverted = convert_expression_data(
        expr_df, orthologs, tissue
    )
    all_stats.append(stats)
    all_unconverted[tissue] = unconverted
    all_mappings[tissue] = mapping

    # Save converted expression
    output_file = os.path.join(output_dir, f'{tissue}_celltype_avg_expression_human.csv')
    expr_converted.to_csv(output_file)
    print(f"  ✓ Saved to {output_file}")

    # Also convert detection rate if available
    detection_file = os.path.join(expr_dir, f'{tissue}_celltype_detection_rate.csv')
    if os.path.exists(detection_file):
        detection_df = load_expression_data(detection_file)
        detection_converted, _, _, _ = convert_expression_data(
            detection_df, orthologs, f"{tissue} (detection rate)"
        )
        detection_output = os.path.join(output_dir, f'{tissue}_celltype_detection_rate_human.csv')
        detection_converted.to_csv(detection_output)
        print(f"  ✓ Saved detection rate to {detection_output}")
    
# Print the current date and time in a specific format
print(now.strftime("%Y-%m-%d %H:%M:%S"))  

Current date and time : 
2025-12-27 16:18:07

Step 3: Converting expression data...
Loading ../R/processed/drgn_celltype_avg_expression.csv...
  Loaded 112469 genes × 18 cell types

Converting drgn expression data...
  Merged 1732 duplicate human genes
  Mouse genes: 112469
  Converted: 18640 (16.6%)
  Unconverted: 93829 (83.4%)
  Final human genes: 16908 (15.0% of original)
  ✓ Saved to converted_expression/drgn_celltype_avg_expression_human.csv
Loading ../R/processed/drgn_celltype_detection_rate.csv...
  Loaded 114660 genes × 18 cell types

Converting drgn (detection rate) expression data...
  Merged 3251 duplicate human genes
  Mouse genes: 112469
  Converted: 18640 (16.6%)
  Unconverted: 93829 (83.4%)
  Final human genes: 16908 (15.0% of original)
  ✓ Saved detection rate to converted_expression/drgn_celltype_detection_rate_human.csv
Loading ../R/processed/drgnn_celltype_avg_expression.csv...
  Loaded 112469 genes × 7 cell types

Converting drgnn expression data...
  Merged 1732 du

In [7]:
# Step 3.5: Deduplicate across tissues
print("\nStep 3.5: Deduplicating genes across tissues...")

# Combine all converted expression data
all_expr_dfs = []
for tissue in tissues:
    output_file = os.path.join(output_dir, f'{tissue}_celltype_avg_expression_human.csv')
    if os.path.exists(output_file):
        expr_df = pd.read_csv(output_file, index_col=0)
        all_expr_dfs.append(expr_df)

if all_expr_dfs:
    # Concatenate all tissues
    expr_combined = pd.concat(all_expr_dfs, axis=1)
    
    # Deduplicate by taking mean across duplicate human genes
    n_before = len(expr_combined)
    expr_combined = expr_combined.groupby(level=0).mean()
    n_after = len(expr_combined)
    n_duplicates = n_before - n_after
    
    print(f"  Before deduplication: {n_before} genes")
    print(f"  After deduplication: {n_after} genes")
    print(f"  Merged: {n_duplicates} duplicate human genes")
    
    # Save deduplicated combined file
    combined_output = os.path.join(output_dir, 'all_tissues_combined_human.csv')
    expr_combined.to_csv(combined_output)
    print(f"  ✓ Saved combined deduplicated file to {combined_output}")


Step 3.5: Deduplicating genes across tissues...
  Before deduplication: 16916 genes
  After deduplication: 16916 genes
  Merged: 0 duplicate human genes
  ✓ Saved combined deduplicated file to converted_expression/all_tissues_combined_human.csv


In [8]:
# Step 4: Generate report
print("\nStep 4: Generating conversion report...")
stats_df = generate_conversion_report(all_stats, all_unconverted, all_mappings, report_dir)

print("\n" + "="*80)
print("CONVERSION COMPLETE!")
print("="*80)
print(f"\nNext steps:")
print(f"1. Review conversion report in: {report_dir}/")
print(f"2. Use converted files from: {output_dir}/")
print(f"3. Update your multiplex script to use '_human.csv' files")
print(f"4. Run: python build_multiplex_interactome.py --expr-dir {output_dir}")
print("="*80)


Step 4: Generating conversion report...

CONVERSION REPORT

Per-Tissue Summary:
tissue  n_mouse_genes  n_converted  n_unconverted  n_human_genes  conversion_rate  retention_rate
  drgn         112469        18640          93829          16908        16.573456       15.033476
 drgnn         112469        18640          93829          16908        16.573456       15.033476
   tgn          67033        18912          48121          16887        28.212970       25.192070
  tgnn          67033        18912          48121          16887        28.212970       25.192070

Overall Statistics:
  Total mouse genes (across all tissues): 359004
  Total converted: 75104 (20.9%)
  Total human genes (after deduplication): 67590
  Average retention: 20.1%

Gene Loss by Tissue:
  drgn: 93829 genes lost
  drgnn: 93829 genes lost
  tgn: 48121 genes lost
  tgnn: 48121 genes lost

✓ Report saved to conversion_report/conversion_report.txt
✓ Statistics saved to conversion_report/conversion_statistics.csv
✓ U